# 병가 정책 시각화 — 발표자료용 그림 4종 (v2 revised)

**출력**: `presentations/figures/viz_A1_diff_curves.png`, `viz_A2_diff_by_pwork.png`, `viz_B1_delta_attack.png`, `viz_C1_policy_map.png`

**셀 구조**: 셀0 imports + PARAMS, 셀1 forward 헬퍼 + fit cache, 셀2 A1(차이곡선), 셀3 A2(강도별 차이), 셀4 B1(색=부호), 셀5 C1(라벨 재배치).

In [1]:
# 셀 0 — imports + 고정 파라미터
# 수정 포인트: PARAMS dict
import sys, os, json
from pathlib import Path

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('XLA_FLAGS', '--xla_force_host_platform_device_count=1')

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
FIG_DIR = REPO_ROOT / 'presentations' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# 한글 폰트 (macOS 우선 AppleGothic) + 마이너스 유니코드 대신 ASCII
from matplotlib import font_manager
available = {f.name for f in font_manager.fontManager.ttflist}
for fnt in ['AppleGothic', 'Malgun Gothic', 'NanumGothic']:
    if fnt in available:
        matplotlib.rcParams['font.family'] = fnt
        break
matplotlib.rcParams['axes.unicode_minus'] = False  # ASCII '-' 사용, '□' 방지

from sens_workshare_kappa_v2 import (
    build_setup, point_fit_at_ws, make_shared, run_forward, build_gamma_15,
    build_pi_target, PHI_USHAPE, GAMMA_CENTER, HIRA_AGE_GROUPS,
    P_WORK_BASE, P_SCHOOL_BASE, SEASON_LABEL,
)
from kt_epimodel_hira.calibration.hira_target import HIRA_GROUP_TO_NIMS_WEIGHTED

# 색맹 배려 팔레트
COL_UP = '#B23A48'    # 증가 (빨강)
COL_DN = '#2166AC'    # 감소 (파랑)
COL_ZERO = '#888888'  # 0/무변화 (회색)

PARAMS = dict(
    season=SEASON_LABEL,
    phi_ushape=PHI_USHAPE.tolist(),
    gamma_report=GAMMA_CENTER,
    seasonality_amp=0.9,
    holiday_realloc=1.0, holiday_amp=0.7,
    ve=0.5, sigma_e=0.5, gamma_rec=0.25,
    sigma_channel=[0.15, 0.01, 0.05, 0.15],
    non_work_a_rel=[0.408, 0.085, 0.507],
)
print(f'PARAMS: season={SEASON_LABEL}   font={matplotlib.rcParams["font.family"]}')

def build_hira_matrix():
    H = np.zeros((6, 15))
    for i, ag in enumerate(HIRA_AGE_GROUPS):
        for j, w in HIRA_GROUP_TO_NIMS_WEIGHTED[ag].items():
            H[i, j] = w
    return H
H_MAT = build_hira_matrix()

PARAMS: season=2019-2020   font=['AppleGothic']


In [2]:
# 셀 1 — forward 헬퍼 + fit 캐시 + scenario 캐시
# 수정 포인트: 필요 시 새 시나리오 추가 → SCEN_CACHE 에 저장됨.
SETUP = build_setup()
pop_15 = np.asarray(SETUP['shared_base']['pop_15'])
if pop_15.ndim == 2:
    pop_15_flat = pop_15.sum(axis=1) if pop_15.shape[0] == 15 else pop_15.sum(axis=0)
else:
    pop_15_flat = pop_15
POP_6 = H_MAT @ pop_15_flat
print(f'pop_6 = {[int(p) for p in POP_6]}')

FIT_CACHE = {}
SCEN_CACHE = {}   # (ws, kappa, p_work) → dict
PHI_FULL_J = jnp.asarray(PHI_USHAPE)
GAMMA_15_J = jnp.asarray(build_gamma_15(*GAMMA_CENTER))

def get_fit(work_share):
    if work_share not in FIT_CACHE:
        print(f'  fitting ws={work_share:.3f} ...')
        FIT_CACHE[work_share] = point_fit_at_ws(work_share, SETUP)
        f = FIT_CACHE[work_share]
        print(f'    R0={f["R0"]:.3f}  β={[round(b,4) for b in f["beta_4"]]}')
    return FIT_CACHE[work_share]

def run_scenario(work_share, kappa, p_work, p_school=1.0):
    key = (round(work_share, 4), round(kappa, 4), round(p_work, 4), round(p_school, 4))
    if key in SCEN_CACHE:
        return SCEN_CACHE[key]
    fit = get_fit(work_share)
    beta_4 = np.array(fit['beta_4'])
    shared = make_shared(kappa, SETUP)
    inc, pred = run_forward(beta_4, PHI_FULL_J, GAMMA_15_J,
                            p_school, p_work, shared, SETUP)
    inc_np = np.asarray(inc)
    pred_np = np.asarray(pred)
    tot_15 = inc_np.sum(axis=0)
    inf_6 = H_MAT @ tot_15
    out = dict(
        pred_hira=pred_np, attack_by_age=inf_6 / POP_6,
        infections_by_age=inf_6, total_infections=float(inf_6.sum()),
        beta_4=beta_4, R0=fit['R0'], pi=fit['pi'],
    )
    SCEN_CACHE[key] = out
    return out

print('run_scenario ready.')

pop_6 = [945049, 1310211, 1409651, 9804120, 8383562, 4137871]
run_scenario ready.


In [3]:
# 셀 2 — [A1] 차이 곡선 (병가 - baseline) 연령별
# 무엇: 각 연령 패널에 weekly(sick) - weekly(baseline) 차이 곡선. 양수=빨강 채움/음수=파랑 채움.
# 수정 포인트: WS, KAPPA, P_SICK. 패널 y축 독립.
WS = 0.12; KAPPA = 0.2; P_SICK = 0.4

base = run_scenario(WS, KAPPA, p_work=1.0)
sick = run_scenario(WS, KAPPA, p_work=P_SICK)

fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.5))
weeks = np.arange(base['pred_hira'].shape[0])

for i, ag in enumerate(HIRA_AGE_GROUPS):
    ax = axes[i // 3, i % 3]
    diff = sick['pred_hira'][:, i] - base['pred_hira'][:, i]
    ax.axhline(0, color=COL_ZERO, lw=0.9)
    ax.fill_between(weeks, 0, diff, where=(diff >= 0), color=COL_UP,
                    alpha=0.55, interpolate=True, label='병가로 증가')
    ax.fill_between(weeks, 0, diff, where=(diff < 0), color=COL_DN,
                    alpha=0.55, interpolate=True, label='병가로 감소')
    ax.plot(weeks, diff, color='black', lw=0.9)
    dpct = 100.0 * (sick['attack_by_age'][i] - base['attack_by_age'][i]) \
        / base['attack_by_age'][i]
    sign = '+' if dpct >= 0 else ''
    ax.set_title(f'{ag}   총 Δattack = {sign}{dpct:.1f}%', fontsize=11)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8, loc='best', framealpha=0.9)
    if i // 3 == 1:
        ax.set_xlabel('epidemic week')
    if i % 3 == 0:
        ax.set_ylabel('주간 발생 차이 (병가 - baseline)')

fig.suptitle(
    f'[A1] 병가에 의한 주간 발생 변화 (병가 - baseline)   '
    f'work_share={WS}, κ={KAPPA}, p_work {P_WORK_BASE}→{P_SICK}   {SEASON_LABEL}',
    fontsize=12.5, fontweight='bold', y=0.995,
)
fig.tight_layout(rect=[0, 0, 1, 0.965])
out = FIG_DIR / 'viz_A1_diff_curves.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved {out}')

  fitting ws=0.120 ...


    R0=2.156  β=[0.0355, 0.0164, 0.0116, 0.0731]


saved /Users/hwcho/Documents/python/NIMS/kt_epimodel_hira/presentations/figures/viz_A1_diff_curves.png


/var/folders/p6/bh0h2f651z556fbmr892wdl80000gn/T/ipykernel_51997/3744899539.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# 셀 3 — [A2] 병가 강도별 차이 곡선 (p_work별, baseline 대비)
# 무엇: 각 연령 패널에 (p_work=0.7/0.4/0.2) - baseline 차이곡선 3개. 강할수록 진한 빨강.
# 수정 포인트: P_WORK_GRID. baseline은 y=0 기준선으로만 표시.
WS = 0.12; KAPPA = 0.2
P_WORK_GRID = [0.7, 0.4, 0.2]   # 병가 강도(약→강)

base = run_scenario(WS, KAPPA, p_work=1.0)
runs = {pw: run_scenario(WS, KAPPA, p_work=pw) for pw in P_WORK_GRID}

fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.5))
weeks = np.arange(base['pred_hira'].shape[0])

# 빨강 계열 단일-색상 그라데이션 (약→강)
red_cmap = plt.get_cmap('Reds')
shades = [red_cmap(0.45), red_cmap(0.65), red_cmap(0.88)]

for i, ag in enumerate(HIRA_AGE_GROUPS):
    ax = axes[i // 3, i % 3]
    ax.axhline(0, color=COL_ZERO, lw=0.9,
                label='baseline (p_work=1.0)' if i == 0 else None)
    for pw, col in zip(P_WORK_GRID, shades):
        diff = runs[pw]['pred_hira'][:, i] - base['pred_hira'][:, i]
        ax.plot(weeks, diff, color=col, lw=2, label=f'p_work={pw}')
    ax.set_title(f'{ag}', fontsize=11)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8, loc='best', framealpha=0.9)
    if i // 3 == 1:
        ax.set_xlabel('epidemic week')
    if i % 3 == 0:
        ax.set_ylabel('주간 발생 차이 (해당 p_work - baseline)')

fig.suptitle(
    f'[A2] 병가 강도별 발생 변화 (p_work별, baseline 대비)   '
    f'work_share={WS}, κ={KAPPA}   빨강 진할수록 강한 병가',
    fontsize=12.5, fontweight='bold', y=0.995,
)
fig.tight_layout(rect=[0, 0, 1, 0.965])
out = FIG_DIR / 'viz_A2_diff_by_pwork.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved {out}')

saved /Users/hwcho/Documents/python/NIMS/kt_epimodel_hira/presentations/figures/viz_A2_diff_by_pwork.png


/var/folders/p6/bh0h2f651z556fbmr892wdl80000gn/T/ipykernel_51997/1377294695.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# 셀 4 — [B1] 연령별 Δattack 막대. 색=부호(빨/파), work_share=그룹 내 위치.
# 무엇: 연령 6군, 각 연령에 ws 3개 막대 (왼→오: ws 0.06/0.12/0.24). 색은 오직 부호.
# 수정 포인트: WS_LIST, KAPPA_B1.
WS_LIST = [0.06, 0.12, 0.24]
KAPPA_B1 = 0.4; P_SICK = 0.4

delta_data = {}
for ws in WS_LIST:
    b = run_scenario(ws, KAPPA_B1, p_work=1.0)
    s = run_scenario(ws, KAPPA_B1, p_work=P_SICK)
    delta_data[ws] = (s['attack_by_age'] - b['attack_by_age']) * 100.0

fig, ax = plt.subplots(figsize=(12, 6.2))
xs = np.arange(len(HIRA_AGE_GROUPS))
n_ws = len(WS_LIST)
width = 0.26

for k, ws in enumerate(WS_LIST):
    offs = (k - (n_ws - 1) / 2) * width
    d = delta_data[ws]
    cols = [COL_UP if v > 0 else COL_DN for v in d]
    ax.bar(xs + offs, d, width, color=cols, edgecolor='black', linewidth=0.7)
    for x, v in zip(xs + offs, d):
        va = 'bottom' if v >= 0 else 'top'
        y_off = 0.06 if v >= 0 else -0.06
        ax.text(x, v + y_off, f'{v:+.2f}', ha='center', va=va, fontsize=7.5)
        # 그룹 내 ws 라벨 (작은 텍스트) — x축 아래에
        ax.text(x, ax.get_ylim()[0] if False else -3.4,
                f'{ws:.2f}', ha='center', va='top', fontsize=6.5,
                color='#333333')

# 부호 legend (2개만)
from matplotlib.patches import Patch
legend_handles = [
    Patch(color=COL_UP, label='병가로 감염 증가 (+)'),
    Patch(color=COL_DN, label='병가로 감염 감소 (-)'),
]
ax.legend(handles=legend_handles, fontsize=10, loc='upper right')

ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(xs)
ax.set_xticklabels(HIRA_AGE_GROUPS, fontsize=11)
ax.set_ylabel('Δ attack rate (병가 - baseline)  [%pt]', fontsize=11)
# 라벨 배치를 위해 y축 하한 확장
yl = ax.get_ylim()
ax.set_ylim(min(yl[0], -3.6), yl[1] * 1.05 if yl[1] > 0 else yl[1])

ax.set_title(
    f'[B1] 연령별 attack rate 변화 (병가 p_work {P_WORK_BASE}->{P_SICK}, κ={KAPPA_B1})\n'
    f'색=부호 (빨강=증가/파랑=감소)  |  각 연령의 3막대 = ws 0.06 / 0.12 / 0.24 (왼→오)',
    fontsize=12,
)
ax.grid(alpha=0.3, axis='y')
fig.tight_layout()
out = FIG_DIR / 'viz_B1_delta_attack.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved {out}')

  fitting ws=0.060 ...


    R0=2.163  β=[0.0356, 0.0077, 0.0118, 0.0733]


  fitting ws=0.240 ...


    R0=2.138  β=[0.0364, 0.0376, 0.0113, 0.0718]


saved /Users/hwcho/Documents/python/NIMS/kt_epimodel_hira/presentations/figures/viz_B1_delta_attack.png


/var/folders/p6/bh0h2f651z556fbmr892wdl80000gn/T/ipykernel_51997/1640902955.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# 셀 5 — [C1] 정책지도 히트맵. Italy/B/A 라벨 및 β 하한 라벨을 히트맵 안쪽 하단에 배치.
# 무엇: 36 combo, x=work_share, y=κ, 셀값=병가 averted% (양수=효과=빨강, 음수=역효과=파랑).
FULL_JSON = REPO_ROOT / 'outputs' / 'eda' / 'sens_workshare_full.json'
if not FULL_JSON.exists():
    raise FileNotFoundError(f'{FULL_JSON} 없음')
full = json.load(open(FULL_JSON))
combos = [c for c in full['combos'] if c.get('policy')]
ws_arr = sorted({c['work_share'] for c in combos})
k_arr = sorted({c['kappa'] for c in combos})

M = np.full((len(k_arr), len(ws_arr)), np.nan)
BW = np.full(len(ws_arr), np.nan)
for c in combos:
    i = k_arr.index(c['kappa']); j = ws_arr.index(c['work_share'])
    M[i, j] = c['policy']['averted_sick_total_pct']
    if c.get('fit') and c['fit'].get('beta_4'):
        BW[j] = c['fit']['beta_4'][1]

TARGETS = [('Italy', 0.033), ('B', 0.17), ('A', 0.29)]
BETA_FLOOR = 0.05

fig, ax = plt.subplots(figsize=(13.5, 5.4))
vmax = np.nanmax(np.abs(M))
cmap = plt.get_cmap('RdBu_r')
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im = ax.imshow(M, aspect='auto', cmap=cmap, norm=norm, origin='lower')
for i in range(len(k_arr)):
    for j in range(len(ws_arr)):
        v = M[i, j]
        if np.isfinite(v):
            col = 'white' if abs(v) > vmax * 0.55 else 'black'
            ax.text(j, i, f'{v:+.1f}', ha='center', va='center',
                    fontsize=8.5, color=col)

# x tick 라벨: 문헌 target 값은 볼드 + 색 강조
target_ws = {round(t, 3): name for name, t in TARGETS}
ax.set_xticks(range(len(ws_arr)))
labels = []
label_colors = []
for w in ws_arr:
    labels.append(f'{w:.2f}')
    # 문헌 근사 지점(±0.005) 은 색 강조
    hit = None
    for name, t in TARGETS:
        if abs(w - t) < 0.006:
            hit = name; break
    label_colors.append('#B23A48' if hit else 'black')
ax.set_xticklabels(labels, rotation=45, fontsize=9)
for tick, col in zip(ax.get_xticklabels(), label_colors):
    tick.set_color(col)
    if col != 'black':
        tick.set_fontweight('bold')

ax.set_yticks(range(len(k_arr)))
ax.set_yticklabels([f'κ={k}' for k in k_arr], fontsize=10)
ax.set_xlabel('work_share (= π_work pin)', fontsize=11, labelpad=32)

# 문헌 target 세로선 (히트맵 안쪽) + 라벨은 히트맵 하단 안쪽 (title 침범 X)
for name, tgt in TARGETS:
    if tgt < ws_arr[0] - 0.005 or tgt > ws_arr[-1] + 0.005:
        continue
    j = float(np.interp(tgt, ws_arr, np.arange(len(ws_arr))))
    ax.axvline(j, color='#333', ls='-', lw=1.0, alpha=0.55)
    # 히트맵 하단(y=-0.42) 안쪽에 배지 형태로 표기
    ax.text(j, -0.48, name, ha='center', va='top',
            fontsize=9, fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.2', fc='#B23A48', ec='none'))

# β_work floor 도달 지점 (녹색 파선) + 라벨 위쪽 안쪽
above = BW >= BETA_FLOOR
if above.any():
    first_j = int(np.argmax(above))
    if first_j > 0 and np.isfinite(BW[first_j-1]):
        j_interp = first_j - 1 + (BETA_FLOOR - BW[first_j-1]) / \
            (BW[first_j] - BW[first_j-1])
    else:
        j_interp = float(first_j)
    ws_interp = float(np.interp(j_interp, np.arange(len(ws_arr)), ws_arr))
    ax.axvline(j_interp, color='#2E8B57', ls='--', lw=1.8, alpha=0.9)
    # 라벨: 히트맵 왼쪽 중간(κ=0.4 라인, title/컬러바 침범 방지)
    ax.text(j_interp - 0.35, 1.0,
            f'β_work={BETA_FLOOR:.2f} 도달선\n(ws≈{ws_interp:.2f})',
            ha='right', va='center', fontsize=8.5, color='white',
            fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.25', fc='#2E8B57', ec='none', alpha=0.92))

# Title (2줄), 히트맵과 여유 확보
ax.set_title(
    f'[C1] 병가 averted %  정책지도 (work_share × κ, {SEASON_LABEL})\n'
    f'빨강 = 감염 감소 (병가 효과)   파랑 = 감염 증가 (역효과)   0 = 흰색',
    fontsize=12, pad=14,
)
cbar = plt.colorbar(im, ax=ax, fraction=0.032, pad=0.02)
cbar.set_label('averted %  ( + 감소 / - 증가 )', fontsize=10)

fig.tight_layout()
out = FIG_DIR / 'viz_C1_policy_map.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved {out}')

saved /Users/hwcho/Documents/python/NIMS/kt_epimodel_hira/presentations/figures/viz_C1_policy_map.png


/var/folders/p6/bh0h2f651z556fbmr892wdl80000gn/T/ipykernel_51997/3727531595.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# 셀 6 — [B3] work_share에 따른 연령별 Δattack 곡선 (κ별 3패널)
# 무엇: sens_workshare_full.json 로부터 연령 6곡선 x work_share (0.03~0.36). κ 3패널.
# 수정 포인트: FULL_JSON 경로. 색은 성인=파랑계열, 아동=빨강계열.
FULL_JSON = REPO_ROOT / 'outputs' / 'eda' / 'sens_workshare_full.json'
full = json.load(open(FULL_JSON))

# combos 에서 policy 가 있으면 baseline/sick 감염 카운트에서 Δattack 유도.
# 없으면 averted_sick_by_age_pct (감염 카운트 % 변화, HIRA count 기준) 사용.
# 여기선 attack rate 대신 "감염 %변화" 로 대체(둘 다 방향은 동일; scale 만 다름).
# 실제 attack rate 는 redistribution 스크립트에만 저장돼 있어, 사용 가능 시 우선.
ws_arr = sorted({c['work_share'] for c in full['combos'] if c.get('policy')})
k_arr = sorted({c['kappa'] for c in full['combos'] if c.get('policy')})
AGE_COLORS = {
    '0-5':    '#7B2CBF',      # 보라 (아동 미취학)
    '6-11':   '#B23A48',      # 빨강 (학령기)
    '12-17':  '#E76F51',      # 주황 (청소년)
    '18-44':  '#2166AC',      # 진파랑 (청년 성인)
    '45-64':  '#4A90D9',      # 파랑 (중장년 성인)
    '65+':    '#525252',      # 회색 (노인)
}
TARGETS = [('Italy', 0.033), ('B', 0.17), ('A', 0.29)]

fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.2), sharey=True)
for ai, k in enumerate(k_arr):
    ax = axes[ai]
    for ag in HIRA_AGE_GROUPS:
        ys = []
        for ws in ws_arr:
            rec = next((c for c in full['combos']
                        if c['work_share']==ws and c['kappa']==k and c.get('policy')), None)
            if rec is None:
                ys.append(np.nan); continue
            # averted_sick_by_age_pct : % of baseline infections averted → sign convention:
            #   양수 = 감염 감소, 음수 = 감염 증가.  Δattack (sick − base) 방향은 반대.
            #   여기선 "-averted" 를 Δattack 대응(양수=병가로 증가)으로 표시.
            v = -rec['policy']['averted_sick_by_age_pct'][ag]
            ys.append(v)
        ax.plot(ws_arr, ys, 'o-', color=AGE_COLORS[ag], lw=1.8, ms=3.5, label=ag)
    ax.axhline(0, color='black', lw=0.8)
    for name, tgt in TARGETS:
        ax.axvline(tgt, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('work_share')
    ax.set_title(f'κ = {k}', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)
    if ai == 0:
        ax.set_ylabel('연령별 감염 변화 %  (병가 - baseline, 양수=증가)')
        ax.legend(fontsize=8, loc='best', ncol=2, framealpha=0.9)

# 문헌 라벨을 x축 아래
for ax in axes:
    ymin = ax.get_ylim()[0]
    for name, tgt in TARGETS:
        ax.text(tgt, ymin, name, ha='center', va='top', fontsize=8,
                color='#555', fontweight='bold')

fig.suptitle(
    '[B3] work_share에 따른 연령별 감염 변화 — 재분배의 전개',
    fontsize=13, fontweight='bold', y=0.995,
)
fig.tight_layout(rect=[0, 0.02, 1, 0.96])
out = FIG_DIR / 'viz_B3_delta_by_workshare.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved {out}')


saved /Users/hwcho/Documents/python/NIMS/kt_epimodel_hira/presentations/figures/viz_B3_delta_by_workshare.png


/var/folders/p6/bh0h2f651z556fbmr892wdl80000gn/T/ipykernel_51997/2045518602.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# 셀 7 — [C3] 문헌 3점 병가 효과 비교 막대 (Italy/B/A × κ)
# 무엇: sens_workshare_full.json 에서 문헌 target 근접 work_share × κ 3개의 averted%.
# 색: 양수(감염 감소=효과)=teal(#0E7C7B), 음수(역효과)=빨강(#B23A48).
TARGET_WS = [('Italy', 0.033), ('B', 0.17), ('A', 0.29)]
KAPPA_GRID_C3 = [0.2, 0.4, 0.6]

def nearest_ws(target):
    return min(ws_arr, key=lambda w: abs(w - target))

target_actual = {name: nearest_ws(t) for name, t in TARGET_WS}
print('nearest ws:', target_actual)

data = {}   # (name, kappa) → averted%
for name, tgt in TARGET_WS:
    ws_used = target_actual[name]
    for k in KAPPA_GRID_C3:
        rec = next((c for c in full['combos']
                    if c['work_share']==ws_used and c['kappa']==k
                    and c.get('policy')), None)
        data[(name, k)] = rec['policy']['averted_sick_total_pct'] if rec else np.nan

fig, ax = plt.subplots(figsize=(10.5, 5.6))
COL_POS = '#0E7C7B'
COL_NEG = '#B23A48'
n_grp = len(TARGET_WS)
n_k = len(KAPPA_GRID_C3)
width = 0.24
xs = np.arange(n_grp)
for ki, k in enumerate(KAPPA_GRID_C3):
    offs = (ki - (n_k - 1)/2) * width
    vals = [data[(name, k)] for name, _ in TARGET_WS]
    cols = [COL_POS if v > 0 else COL_NEG for v in vals]
    ax.bar(xs + offs, vals, width, color=cols,
            edgecolor='black', linewidth=0.7)
    for x, v in zip(xs + offs, vals):
        va = 'bottom' if v >= 0 else 'top'
        y_off = 0.5 if v >= 0 else -0.5
        ax.text(x, v + y_off, f'{v:+.1f}%', ha='center', va=va,
                fontsize=9, fontweight='bold')
    # κ 라벨 그룹 위/아래
    for x in xs + offs:
        ax.text(x, -30 if False else ax.get_ylim()[0] if False else 0.5,
                f'κ={k}', ha='center', fontsize=6.5, color='#666',
                alpha=0)  # 미표시, 대체로 아래에 별도 라벨

# 그룹 아래 소라벨: κ 정보
ymin = ax.get_ylim()[0] if ax.get_ylim()[0] < -3 else -20
ax.set_ylim(min(-20, ax.get_ylim()[0]*1.05), max(35, ax.get_ylim()[1]*1.1))
for ki, k in enumerate(KAPPA_GRID_C3):
    offs = (ki - (n_k - 1)/2) * width
    for x in xs:
        ax.text(x + offs, ax.get_ylim()[0] * 0.96,
                f'κ={k}', ha='center', va='top', fontsize=7, color='#333')

from matplotlib.patches import Patch
legend_handles = [
    Patch(color=COL_POS, label='병가로 감염 감소 (효과 있음)'),
    Patch(color=COL_NEG, label='병가로 감염 증가 (역효과)'),
]
ax.legend(handles=legend_handles, fontsize=10, loc='upper left')

ax.axhline(0, color='black', lw=0.9)
ax.set_xticks(xs)
ax.set_xticklabels(
    [f'{name}\n(ws≈{target_actual[name]:.2f})' for name, _ in TARGET_WS],
    fontsize=11, fontweight='bold',
)
ax.set_ylabel('averted %  (병가 총감염 감소율)', fontsize=11)
ax.set_title(
    '[C3] 문헌 target별 병가 효과 — 어느 값을 믿느냐에 따라\n'
    f'Italy 3.3% / B 17% / A 29%, 각 κ=0.2·0.4·0.6',
    fontsize=12,
)
ax.grid(alpha=0.3, axis='y')
fig.tight_layout()
out = FIG_DIR / 'viz_C3_literature_targets.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved {out}')


/var/folders/p6/bh0h2f651z556fbmr892wdl80000gn/T/ipykernel_51997/1007616965.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


nearest ws: {'Italy': 0.03, 'B': 0.18, 'A': 0.3}
saved /Users/hwcho/Documents/python/NIMS/kt_epimodel_hira/presentations/figures/viz_C3_literature_targets.png


In [9]:
# 셀 8 — [A3] 전체 유행곡선 (전 연령 합) baseline vs 병가 + 차이곡선
# 무엇: 상단 전 연령 합산 곡선 겹치기, 하단 차이곡선 (fill 부호별).
# 수정 포인트: WS_A3, KAPPA_A3, P_SICK_A3. run_scenario 캐시 재사용.
WS_A3 = 0.12; KAPPA_A3 = 0.2; P_SICK_A3 = 0.4

base = run_scenario(WS_A3, KAPPA_A3, p_work=1.0)
sick = run_scenario(WS_A3, KAPPA_A3, p_work=P_SICK_A3)
total_b = base['pred_hira'].sum(axis=1)
total_s = sick['pred_hira'].sum(axis=1)
weeks = np.arange(total_b.shape[0])
tot_averted = 100.0 * (total_b.sum() - total_s.sum()) / max(total_b.sum(), 1)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                          gridspec_kw={'height_ratios':[1.2, 1]})
ax = axes[0]
ax.plot(weeks, total_b, '-', color='#1f4e79', lw=2.2, label='baseline (p_work=1.0)')
ax.plot(weeks, total_s, '--', color='#B23A48', lw=2.2, label=f'병가 (p_work={P_SICK_A3})')
ax.set_ylabel('전 연령 합 주간 발생 (건)')
ax.legend(fontsize=10)
ax.set_title(
    f'[A3] 전체 유행곡선 — baseline vs 병가 (총량, 전 연령 합)   '
    f'work_share={WS_A3}, κ={KAPPA_A3}   총 averted = {tot_averted:+.2f}%',
    fontsize=12, fontweight='bold',
)
ax.grid(alpha=0.3)

ax = axes[1]
diff = total_s - total_b
ax.axhline(0, color=COL_ZERO, lw=0.9)
ax.fill_between(weeks, 0, diff, where=(diff>=0), color=COL_UP, alpha=0.55,
                interpolate=True, label='병가로 증가 (총량)')
ax.fill_between(weeks, 0, diff, where=(diff<0), color=COL_DN, alpha=0.55,
                interpolate=True, label='병가로 감소 (총량)')
ax.plot(weeks, diff, color='black', lw=0.9)
ax.set_ylabel('전 연령 합 발생 차이 (병가 - baseline)')
ax.set_xlabel('epidemic week')
ax.legend(fontsize=9, loc='best')
ax.grid(alpha=0.3)
fig.tight_layout()
out = FIG_DIR / 'viz_A3_total_curve.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved {out}')


saved /Users/hwcho/Documents/python/NIMS/kt_epimodel_hira/presentations/figures/viz_A3_total_curve.png


/var/folders/p6/bh0h2f651z556fbmr892wdl80000gn/T/ipykernel_51997/1952198260.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# 셀 9 — [E3] 연령 프로파일 φ + R(0) (파라미터만, 시뮬 불필요)
# 무엇: 좌 감수성 U-shape, 우 초기면역 계단형. NIMS-15 x축.
# 수정 포인트: PHI_USHAPE, R0_PROFILE.
PHI_USHAPE_LOCAL = np.array([2.0,1.9,1.7,1.4,1.1,1.0,1.0,1.0,1.0,1.05,1.1,1.2,1.3,1.4,1.5])
R0_PROFILE = np.array([0.10]*4 + [0.30]*6 + [0.45]*3 + [0.65]*2)
NIMS_LABELS = ['0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39',
                '40-44','45-49','50-54','55-59','60-64','65-69','70+']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
ax.plot(range(15), PHI_USHAPE_LOCAL, 'o-', color='#B23A48', lw=2, ms=6)
ax.set_xticks(range(15))
ax.set_xticklabels(NIMS_LABELS, rotation=45, fontsize=8)
ax.set_ylabel('감수성 φ (기준 1.0)')
ax.set_title('φ — 연령별 상대 감수성 (U-shape)', fontsize=11, fontweight='bold')
ax.grid(alpha=0.3)
# 강조 주석
ax.annotate('소아 2.0\n(가장 취약)', xy=(0, 2.0), xytext=(1.6, 2.05),
            fontsize=9, color='#B23A48', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#B23A48'))
ax.annotate('노인 1.5', xy=(14, 1.5), xytext=(11.5, 1.6),
            fontsize=9, color='#B23A48', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#B23A48'))
ax.axhline(1.0, color='gray', ls=':', lw=0.8)
ax.set_ylim(0.9, 2.2)

ax = axes[1]
ax.step(range(15), R0_PROFILE, where='mid', color='#2166AC', lw=2)
ax.fill_between(range(15), 0, R0_PROFILE, step='mid', color='#2166AC', alpha=0.25)
ax.set_xticks(range(15))
ax.set_xticklabels(NIMS_LABELS, rotation=45, fontsize=8)
ax.set_ylabel('초기 면역 보유율 R(0)')
ax.set_title('R(0) — 초기 면역 프로파일 (기존 노출 반영, 계단형)', fontsize=11,
              fontweight='bold')
ax.grid(alpha=0.3)
ax.set_ylim(0, 0.75)
for xi, r in [(1,0.10), (7,0.30), (10,0.45), (13.5,0.65)]:
    ax.text(xi, r + 0.03, f'{r:.2f}', ha='center', fontsize=9, color='#2166AC',
            fontweight='bold')
ax.annotate('노인 0.65\n(기존 노출 많음)', xy=(13.5, 0.65), xytext=(9.5, 0.72),
            fontsize=9, color='#2166AC', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#2166AC'))
ax.annotate('영유아 0.10\n(무면역)', xy=(1.5, 0.10), xytext=(3, 0.06),
            fontsize=9, color='#2166AC', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#2166AC'))

fig.suptitle('[E3] 연령 프로파일 — 감수성 φ 와 초기면역 R(0)',
              fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
out = FIG_DIR / 'viz_E3_age_profiles.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved {out}')


saved /Users/hwcho/Documents/python/NIMS/kt_epimodel_hira/presentations/figures/viz_E3_age_profiles.png


/var/folders/p6/bh0h2f651z556fbmr892wdl80000gn/T/ipykernel_51997/3039504643.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
